# Tic toc toe

In [1]:
import numpy as np

"""

"""

# Game board
board = np.zeros(9, dtype=np.int8)

def display_board(board):
    # display_board = board.reshape(3,3)
    # for loop...

    # Simple solution
    print(f"{board[0]} | {board[1]} | {board[2]}")
    print(f"{board[3]} | {board[4]} | {board[5]}")
    print(f"{board[6]} | {board[7]} | {board[8]}")
    print()

def display_board_characters(board):
    # strBoard = [str(i) for i in board]
    strBoard = np.empty(9, dtype='U1')
    strBoard[np.where(board == 1)[0]] = 'X'
    strBoard[np.where(board == 0)[0]] = ' '
    strBoard[np.where(board == -1)[0]] = 'O'

    # Simple solution
    display_board(board=strBoard)


board[0] = 1
board[1] = 1
board[2] = 1

board[8] = -1

print(f"Game board {board}")

display_board(board=board)
display_board_characters(board=board)

Game board [ 1  1  1  0  0  0  0  0 -1]
1 | 1 | 1
0 | 0 | 0
0 | 0 | -1

X | X | X
  |   |  
  |   | O



In [2]:
def encode_array(arrIndices):
    # Function encodes array of indices into array of ones
    #
    #   arrIndices: Indices array
    #
    rtn = np.zeros(9)
    rtn[arrIndices] = 1
    return rtn

def check_game_complete(board, solutionMatrix):

    depth = np.sum(np.abs(board))   # No. of turns taken

    # Look at results
    scores = solutionMatrix.dot(board)

    if np.max(scores) == 3:     # X has won
        return 10 - depth
    elif np.min(scores) == -3:  # O has won
        return -10 + depth
    elif depth == 9:            # Draw
        return 0

    return None     # Game not finished

# Define solutions as indices of the board    
solutions = [[0, 1, 2], [3, 4, 5], [6, 7, 8],   # Rows
             [0, 3, 6], [1, 4, 7], [2, 5, 8],   # Columns
             [0, 4, 8], [2, 4, 6]]              # Diagonals

# Encode indices into arrays of 1s & 0s to form solutionMatrix
solMat = np.apply_along_axis(encode_array, axis=1, arr=solutions, )

testBoardXWin = [1, 1, 1, 0, -1, 0, -1, 0, -1]
print(check_game_complete(board=testBoardXWin, solutionMatrix=solMat))

testBoardNone = [1, -1, 1, 0, -1, 0, -1, 0, -1]
print(check_game_complete(board=testBoardNone, solutionMatrix=solMat))

testBoardOWin = [-1, 1, 1, 0, -1, 0, -1, 0, -1]
print(check_game_complete(board=testBoardOWin, solutionMatrix=solMat))

testBoardDraw = [1, -1, 1, -1, -1, 1, -1, 1, -1]
print(check_game_complete(board=testBoardDraw, solutionMatrix=solMat))

4
None
-4
0


In [3]:
def random_player(board, XorO):
    """
    Player function - Takes in the current board and return a move.

    random_player: Move is randomly selected

    Args:
        board (array(9, int)): Array of int that represent the current board
        XorO (int): value of 1 or -1, corresponding to X and O respectively

    Returns:
        int: index of move with respect to board
    """
    possibleMoves = np.where(board == 0)[0]
    move = np.random.choice(possibleMoves)
    return move

In [4]:
# Game loop

def playGame(playerX, playerO, solMat, startBoard, startPlayer = 1):

    plyBoard = startBoard
    player = startPlayer

    # Loop taking turns
    for t in range(9):   # Max of 9 turns, while loop would work but this helps debugging
        # Player makes move
        if player == 1:
            plyBoard[playerX(plyBoard, player)] = player
        else: # player == -1
            plyBoard[playerO(plyBoard, player)] = player
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is None:      # Evaluate 'None' first as cannot be used in equality
            player = -player    # Change player
            continue            # Skip to next turn          
        elif result > 0:
            print("Game is won by X (1)")
            display_board_characters(plyBoard)
            break               
        elif result < 0:
            print("Game is won by O (-1)")
            display_board_characters(plyBoard)
            break
        elif result == 0:
            print("Game is a draw")
            display_board_characters(plyBoard)
            break
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)
        
    return None # Will be updated when I need game data for Neural Network

In [5]:
# Random vs Random players

noGames = 5
# Loop number of games
for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGame(playerX=random_player, playerO=random_player, solMat=solMat, startBoard=rndBoard)

Game is won by O (-1)
O | O | O
X | X | O
X | X |  

Game is won by X (1)
  | O |  
X | O | O
X | X | X

Game is won by O (-1)
X | X | O
X | O | O
O |   | X

Game is won by X (1)
O | X | X
O | X | X
X | O | O

Game is a draw
O | X | O
O | X | X
X | O | X



## Move to class structure

In [6]:
# Abstract class
from abc import ABC, abstractmethod

# Blueprint for tic tac toe player
class player(ABC):
    @abstractmethod
    def nextMove(self, board):
        """
        Calculated next move based on current state (board)
        Args:
            board (array(9, int)): Array of int that represent the current board
        
        Returns:
            int: index of move with respect to board
            
        """
        pass

In [7]:
# Random player
class RandomPlayer(player):
    
    def __init__(self):
        pass

    def nextMove(self, board):
        possibleMoves = np.where(board == 0)[0]
        move = np.random.choice(possibleMoves)
        return move

In [47]:
def playGameClass(playerX, playerO, solMat, startBoard, startPlayer = 1):

    plyBoard = startBoard
    player = startPlayer
    result = None

    # Loop taking turns
    for t in range(9):   # Max of 9 turns, while loop would work but this helps debugging
        # Player makes move
        if player == 1:
            plyBoard[playerX.nextMove(plyBoard)] = player
        else: # player == -1
            plyBoard[playerO.nextMove(plyBoard)] = player
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is None:      # Evaluate 'None' first as cannot be used in equality
            player = -player    # Change player
            continue            # Skip to next turn          
        elif result > 0:
            print(f"Game is won by X (1), first move was {startPlayer}")
            display_board_characters(plyBoard)
            break               
        elif result < 0:
            print(f"Game is won by O (-1), first move was {startPlayer}")
            display_board_characters(plyBoard)
            break
        elif result == 0:
            print(f"Game is a draw, first move was {startPlayer}")
            display_board_characters(plyBoard)
            break
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)
        
    return result # Will be updated when I need game data for Neural Network

In [9]:
player1X = RandomPlayer()
player2O = RandomPlayer()

for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard)

Game is a draw, first move was 1
X | O | O
O | X | X
X | X | O

Game is won by O (-1), first move was 1
X | O | O
  | O | X
X | O | X

Game is won by O (-1), first move was 1
X |   | O
O | O | X
O | X | X

Game is won by X (1), first move was 1
O | X | O
X | X | X
O | X | O

Game is won by O (-1), first move was 1
  |   | O
  | O | X
O | X | X



In [10]:
# MaxMin player
class MinMaxPlayer(player):
    
    def __init__(self, playerType, solutionMatrix):
        self.playerType = playerType
        self.solutionMatrix = solutionMatrix
    
    def _minmax(self, board, currentPlayer):
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=board, solutionMatrix=self.solutionMatrix)
    
        if result is None:
            
            # Game not concluded, recursively try next move with _minmax
            bestScore = -np.inf #* currentPlayer 
            possMoves = np.where(board == 0)[0]

            for move in possMoves:

                testBoard = board.copy()
                # Try a move
                testBoard[move] = currentPlayer

                # Recursive call of _minmax
                # Note: 'currentPlayer *' ensure bestScore is always a positive value
                bestScore = max(bestScore, 
                                (currentPlayer * self._minmax(board = testBoard, 
                                                              currentPlayer = -currentPlayer)))

            return bestScore * currentPlayer    
        elif result > 0:
            return result             
        elif result < 0:
            return result
        elif result == 0:
            return result
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)

    def nextMove(self, board):
        
        possibleMoves = np.where(board == 0)[0]
        bestValue = -np.inf
        bestMove = possibleMoves[0]
        scoreTracker = np.ones(9) - 100

        for move in possibleMoves:
            testBoard = board.copy()
            
            # Try a move
            testBoard[move] = self.playerType

            # Score move
            # Note: Current player is updated with '-'
            moveValue = self.playerType * self._minmax(board = testBoard, currentPlayer = -self.playerType)

            scoreTracker[move] = moveValue

            # Update best move
            if moveValue > bestValue:
                bestMove = move
                bestValue = moveValue

        # Add some randomness to player were possible
        # If a draw is the highest score, the draw value selected is random
        if bestValue == 0:
            possMoves = np.where(scoreTracker == 0)[0]
            bestMove = np.random.choice(possMoves)
            
        # print(scoreTracker)

        return bestMove
    
    

In [11]:
pvalue = -1

player2X = MinMaxPlayer(playerType = pvalue, solutionMatrix=solMat)

testBoard = np.array([1, -1, 1, 0, 0, 0, 0, 0, 0], dtype=np.int8)

move = player2X.nextMove(testBoard)

display_board_characters(testBoard)
print(move)
testBoard[move] = pvalue
display_board_characters(testBoard)



X | O | X
  |   |  
  |   |  

4
X | O | X
  | O |  
  |   |  



In [12]:
player1X = MinMaxPlayer(playerType = 1, solutionMatrix=solMat)
# player1X = RandomPlayer()
player2O = MinMaxPlayer(playerType = -1, solutionMatrix=solMat)
# player2O = RandomPlayer()

for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard, startPlayer=1)

# testBoard = np.array([0, -1, 0, 0, 0, 0, 0, 1, 0], dtype=np.int8)
# playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=testBoard, startPlayer=1)

Game is a draw, first move was 1
X | O | X
X | O | O
O | X | X



## Neural network :)

In [13]:
# minmax a board
def dataGenMinmax(board, currentPlayer, solutionMatrix):
        
    # Evaluate game (has anyone won?)
    result = check_game_complete(board=board, solutionMatrix=solutionMatrix)

    if result is None:
        
        # Game not concluded, recursively try next move with _minmax
        bestScore = -np.inf #* currentPlayer 
        possMoves = np.where(board == 0)[0]

        for move in possMoves:

            testBoard = board.copy()
            # Try a move
            testBoard[move] = currentPlayer

            # Recursive call of _minmax
            # Note: 'currentPlayer *' ensure bestScore is always a positive value
            bestScore = max(bestScore, 
                            (currentPlayer * dataGenMinmax(board = testBoard, 
                                                            currentPlayer = -currentPlayer,
                                                            solutionMatrix=solutionMatrix)))

        return bestScore * currentPlayer    # minmax score
    return result   # End game score (win, draw, lose)

def dataGenNextMoveScore(board, solutionMatrix, player):
        
        currentPlayer = player
        possibleMoves = np.where(board == 0)[0]
        
        # print(f"currentPlayer: {currentPlayer}")
        # print(f"possibleMoves: {possibleMoves}")
        # print(f"board: {board}")

        scoreTracker = np.zeros(9) - 20 # move not available = score -20

        for move in possibleMoves:
            testBoard = board.copy()
            
            # Try a move
            testBoard[move] = currentPlayer

            # Score move
            moveValue = currentPlayer * dataGenMinmax(board = testBoard, 
                                                      currentPlayer = -currentPlayer, 
                                                      solutionMatrix=solutionMatrix)
            # Update tracker
            scoreTracker[move] = moveValue

        return scoreTracker


# Test data generator

testBoard = np.array([0, -1, 0, 0, 0, 0, 0, 1, 0], dtype=np.int8)

testResult = dataGenNextMoveScore(board=testBoard, solutionMatrix=solMat, player=1)  

display_board_characters(testBoard)
print(testResult)

  | O |  
  |   |  
  | X |  

[  0. -20.   0.   0.   0.   0.   0. -20.   0.]


In [24]:
# Data generation

def dataGeneration(playerX, playerO, solutionMatrix, startBoard, startPlayer = 1):

    # dataX = np.empty([9, 9], np.int8)
    # dataY = np.empty([9, 9], np.int8)

    dataX = []
    dataY = []

    # Start a game from a board
    # For each move: record the move (X) & record the minmax score (Y) for training data

    plyBoard = startBoard.copy()
    player = startPlayer

    # Loop through the maximum of 9 moves
    for t in range(9):
        # update board with next move
        if player == 1:
            plyBoard[playerX.nextMove(plyBoard)] = player
        else: # player == -1
            plyBoard[playerO.nextMove(plyBoard)] = player

        player = -player

        # dataX[t, :] = plyBoard
        # dataY[t, :] = dataGenNextMoveScore(board=plyBoard, 
        #                                   solutionMatrix=solutionMatrix, 
        #                                   player=player)

        dataX.append(plyBoard.tolist())
        dataY.append(dataGenNextMoveScore(board=plyBoard, 
                                          solutionMatrix=solutionMatrix, 
                                          player=player).tolist())
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is not None:
            # if t < 8:
            #     dataX = np.delete(dataX, [t, 8], axis=1)
            #     dataY = np.delete(dataY, [t, 8], axis=1)
            return (dataX, dataY)
    

# Test data generation

trainDataX = []
trainDataY = []
tempX = []
tempY = []

player1X = RandomPlayer()
player2O = MinMaxPlayer(playerType = -1, solutionMatrix=solMat)
# player2O = RandomPlayer()

noGames = 5000

for n in range(noGames):
    startBoard = np.zeros(9, dtype=int)
    tempX, tempY = dataGeneration(player1X, player2O, solMat, startBoard)
    trainDataX.extend(tempX)
    trainDataY.extend(tempY)
    
    if n % 100 == 0:
        print(f"At game {n}")

trainDataX = np.array(trainDataX)
trainDataY = np.array(trainDataY)

print(f"Training data X: {trainDataX.shape}")
print(f"Training data Y: {trainDataY.shape}")

# Would be more efficient to write data directly to file :)
fnTrainDataX = 'tranDataX'
fnTrainDataY = 'trainDataY'
np.save(fnTrainDataX, trainDataX)
np.save(fnTrainDataY, trainDataY)


Training data X: (34967, 9)
Training data Y: (34967, 9)


In [45]:
# NN player

class nn_v1_player(player):

    def __init__(self, model):
        self.model = model

    def nextMove(self, board):

        # Get prediction from model for current board
        prediction = self.model.predict(board.reshape(1,9))

        # Convert prediction into move and return
        move = np.argmax(prediction)    # return the FIRST best move

        return move

In [68]:
import tensorflow as tf

# Import data
# np.load(fnTrainDataX, trainDataX)
# np.load  (fnTrainDataY, trainDataY)

# trainDataX_unique = np.unique(trainDataX, axis=0)
# trainDataY_unique = np.unique(trainDataY, axis=0)

# one-hot y data
maxIndices = np.argmax(trainDataY, axis=1)
trainDataY_oneHot = tf.one_hot(maxIndices, 9)

#  Build model
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(81, activation='relu'),
    tf.keras.layers.Dense(45, activation='relu'),
    tf.keras.layers.Dense(9)
])

# predictions = model(trainDataX[:1]).numpy()

# tf.nn.softmax(predictions).numpy()

loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

# loss_fn(trainDataY[:1], predictions).numpy()

model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

# Train model
model.fit(trainDataX, trainDataY_oneHot, epochs=10)
# model.fit(trainDataX, trainDataY, epochs=10)



Epoch 1/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 690us/step - accuracy: 0.6215 - loss: 1.1675
Epoch 2/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 665us/step - accuracy: 0.8999 - loss: 0.3828
Epoch 3/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 670us/step - accuracy: 0.9600 - loss: 0.1769
Epoch 4/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 672us/step - accuracy: 0.9800 - loss: 0.0984
Epoch 5/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 662us/step - accuracy: 0.9895 - loss: 0.0616
Epoch 6/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 666us/step - accuracy: 0.9941 - loss: 0.0402
Epoch 7/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 683us/step - accuracy: 0.9965 - loss: 0.0274
Epoch 8/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 669us/step - accuracy: 0.9975 - loss: 0.0197
Epoch 9/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 714us/step - accuracy: 0.9984 - loss: 0.0144
Epoch 10/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 697us/step - accuracy: 0.9989 - loss: 0.0105


In [53]:
# Evaluate model

# player1X = MinMaxPlayer(playerType = 1, solutionMatrix=solMat)
player1X = RandomPlayer()
player2O = nn_v1_player(model=model)

countResult = [3]

tempCount = np.zeros(3, dtype=np.int16)

for n in range(50):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    result = playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard, startPlayer=1)

    if result > 0:
        tempCount[0] += 1
        # print("Game is won by X (1)")
        # display_board_characters(plyBoard)               
    elif result < 0:
        tempCount[1] += 1
        # print("Game is won by O (-1)")
        # display_board_characters(plyBoard)
    else:
        tempCount[2] += 1
        # print("Game is a draw")
        # display_board_characters(plyBoard)

print(f"counts: {tempCount}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Game is won by O (-1), first move was 1
X | X | O
O | O | O
X | X |  

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Game is won by O (-1), first move was 1
O | O | O
X |   | X
O | X | X

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Game is a draw, first move was 1
O | X | O
O | X | X
X | O | X

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Game is won by O (-1), first move was 1
O | O | O
  |   | X
X |   | X

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Game is won by O (-1), first move was 1